In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction --extra-index-url https://download.easydiffraction.org/pypi/simple/

# Single-crystal fit: Pr₂NiO₄, 5C2

Fit the layered nickelate Pr₂NiO₄ to constant-wavelength neutron single-crystal data from 5C2 at
LLB, with anisotropic displacements.

The project is the catalogue case `sc-neut-cwl_pr2nio4-5c2`, the one the command line and the
app's Examples open.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Single</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">sc</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">cwl</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal</span>

In [ ]:
import easydiffraction as edi

## Load the project

In [ ]:
project_path = edi.download_data('sc-neut-cwl_pr2nio4-5c2', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['pr2nio4']
experiment = project.experiments['llb_5c2']

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [ ]:
for parameter in project.free_parameters:
    parameter.free = False

In [ ]:
project.display.pattern('llb_5c2')

## Perform the analysis

A single crystal has no background or profile: the stages are the scale, the positions, the
interstitial oxygen's displacement, then the displacement tensors.

### Scale

The scale brings the calculated intensities to the measured ones.

In [ ]:
experiment.linked_structures['pr2nio4'].scale.free = True

In [ ]:
project.display.parameters.free()

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('llb_5c2')

### Positions

The coordinates each site's symmetry leaves free.

In [ ]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('llb_5c2')

### The interstitial displacement

The isotropic displacement of the interstitial oxygen Od.

In [ ]:
structure.atom_sites['Od'].adp_iso.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('llb_5c2')

### Anisotropic displacements

Last, the tensors, each with the components its site symmetry leaves free.

In [ ]:
for tensor in structure.atom_site_aniso:
    for component in (
        tensor.adp_11,
        tensor.adp_22,
        tensor.adp_33,
        tensor.adp_12,
        tensor.adp_13,
        tensor.adp_23,
    ):
        component.free = not component.symmetry_constrained

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('llb_5c2')

In [ ]:
project.display.fit.correlations()

In [ ]:
project.display.structure('pr2nio4')

## Save the project

In [ ]:
project.save_as('sc-neut-cwl_pr2nio4-5c2')
project.report.save_html('sc-neut-cwl_pr2nio4-5c2.html')